In [13]:
#Phase 2 — correspondance d’identifiants : 0 % fait
#ncbi_gene_info et uniprot_idmapping sont encore des fichiers globaux à extraire par taxon via API — pas construit. Sans ça, impossible de relier proprement un GeneID NCBI à une entrée UniProt à un domaine InterPro pour le même gène.

In [14]:
#Cellule 1 — remplace ta cellule actuelle par celle-ci (ajoute juste build_id_map.py à la copie) :
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive"
DRIVE_DIR = ROOT + "/files"
DATA_DIR = DRIVE_DIR + "/data"

for name in ("sources.yaml", "fetch_raw.py", "build_id_map.py", "write_phase_readme.py"):
    assert os.path.exists(f"{DRIVE_DIR}/{name}"), f"{name} absent de {DRIVE_DIR}"
    shutil.copy(f"{DRIVE_DIR}/{name}", ".")
print("Scripts copies depuis", DRIVE_DIR, "| donnees dans", DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Scripts copies depuis /content/drive/MyDrive/files | donnees dans /content/drive/MyDrive/files/data


In [15]:
#Cellule 2 — télécharge les 3 sources UniProt (ID mapping) :
OPTIONS = ""
SOURCES = ["uniprot_idmap_tabacum", "uniprot_idmap_zea_mays", "uniprot_idmap_oryza_sativa"]

import subprocess, shlex, sys
for sid in SOURCES:
    cmd = (f'python -u fetch_raw.py sources.yaml --source {sid} '
           f'--raw-dir "{DATA_DIR}/raw" --manifest "{DATA_DIR}/manifest.csv" --no-disk-check {OPTIONS}')
    print("$", cmd, flush=True)
    p = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for ch in iter(lambda: p.stdout.read(1), ""):
        sys.stdout.write(ch); sys.stdout.flush()
    print("code de sortie :", p.wait(), "\n")

$ python -u fetch_raw.py sources.yaml --source uniprot_idmap_tabacum --raw-dir "/content/drive/MyDrive/files/data/raw" --manifest "/content/drive/MyDrive/files/data/manifest.csv" --no-disk-check 

[uniprot_idmap_tabacum] 1 fichiers, ~0 Mo declares
    - uniprot_idmap_nicotiana_tabacum_4097_reviewed.tsv.gz  ~0.2 Mo
  > uniprot_idmap_nicotiana_tabacum_4097_reviewed.tsv.gz ... ok (0.0 Mo)
code de sortie : 0 

$ python -u fetch_raw.py sources.yaml --source uniprot_idmap_zea_mays --raw-dir "/content/drive/MyDrive/files/data/raw" --manifest "/content/drive/MyDrive/files/data/manifest.csv" --no-disk-check 

[uniprot_idmap_zea_mays] 1 fichiers, ~0 Mo declares
    - uniprot_idmap_zea_mays_4577_reviewed.tsv.gz  ~0.3 Mo
  > uniprot_idmap_zea_mays_4577_reviewed.tsv.gz ... ok (0.0 Mo)
code de sortie : 0 

$ python -u fetch_raw.py sources.yaml --source uniprot_idmap_oryza_sativa --raw-dir "/content/drive/MyDrive/files/data/raw" --manifest "/content/drive/MyDrive/files/data/manifest.csv" --no-disk-ch

In [16]:
#Cellule 3 — vérifie les vrais noms de colonnes avant de faire confiance au script (je ne suis pas certain à 100 % que UniProt nomme ses colonnes exactement GeneID/InterPro) :
import gzip, glob
p = glob.glob(f"{DATA_DIR}/raw/nicotiana_tabacum/uniprot_idmap_tabacum/*/*.tsv.gz")[0]
with gzip.open(p, "rt") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 2:
            break

Entry	Reviewed	Entry Name	Protein names	Gene Names	RefSeq	GeneID	InterPro	Ensembl
A0A075QQ08	reviewed	IF4E1_TOBAC	Eukaryotic translation initiation factor 4E-1 (NteIF4E1) (eIF4E-1) (Eukaryotic translation initiation factor 4E2-T) (eIF4E2-T) (eIF-4F 25 kDa subunit) (eIF-4F p26 subunit) (mRNA cap-binding protein)	eIF4E EIF4E2-T T015277 LOC107829212	NP_001313148.1;	107829212;	IPR023398;IPR001040;IPR019770;
A0A1S3X835	reviewed	MBP2C_TOBAC	Protein MICROTUBULE BINDING PROTEIN 2C (NtMBP2C) (Movement protein binding protein 2C) (TMV-MP30 binding protein 2C)	MBP2C LOC107762219	NP_001311679.1 [A0A1S3X835-2];	107762219;	IPR040289;


In [18]:
species_assemblies = [
    ("zea_mays", "ncbi_refseq_zea_b73_v5", "Zm-B73-REFERENCE-NAM-5.0", "uniprot_idmap_zea_mays"),
    ("oryza_sativa", "ncbi_refseq_oryza_irgsp", "IRGSP-1.0", "uniprot_idmap_oryza_sativa"),
    ("nicotiana_tabacum", "ncbi_refseq_nitab_tn90", "Ntab-TN90", "uniprot_idmap_tabacum"),
    ("nicotiana_tabacum", "ncbi_refseq_nitab_asm71507v2", "ASM71507v2_K326", "uniprot_idmap_tabacum"),
]

import glob
for species, ncbi_source, assembly_label, uniprot_source in species_assemblies:
    ft = glob.glob(f"{DATA_DIR}/raw/{species}/{ncbi_source}/*/*_feature_table.txt.gz")
    up = glob.glob(f"{DATA_DIR}/raw/{species}/{uniprot_source}/*/*.tsv.gz")
    assert ft, f"feature_table introuvable pour {ncbi_source}"
    print(f"  (debug) up trouve pour {uniprot_source} :", up)
    cmd = (f'python -u build_id_map.py --feature-table "{ft[0]}" '
           + (f'--uniprot-idmap "{up[0]}" ' if up else '')
           + f'--species {species} --assembly "{assembly_label}" '
           + f'--out "{DATA_DIR}/id_map_{species}_{assembly_label}.csv"')
    print("$", cmd)
    !{cmd}
    print()

  (debug) up trouve pour uniprot_idmap_zea_mays : ['/content/drive/MyDrive/files/data/raw/zea_mays/uniprot_idmap_zea_mays/rolling_2026-10-11/uniprot_idmap_zea_mays_4577_reviewed.tsv.gz']
$ python -u build_id_map.py --feature-table "/content/drive/MyDrive/files/data/raw/zea_mays/ncbi_refseq_zea_b73_v5/Zm-B73-REFERENCE-NAM-5.0_GCF_902167145.1/GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_feature_table.txt.gz" --uniprot-idmap "/content/drive/MyDrive/files/data/raw/zea_mays/uniprot_idmap_zea_mays/rolling_2026-10-11/uniprot_idmap_zea_mays_4577_reviewed.tsv.gz" --species zea_mays --assembly "Zm-B73-REFERENCE-NAM-5.0" --out "/content/drive/MyDrive/files/data/id_map_zea_mays_Zm-B73-REFERENCE-NAM-5.0.csv"
Lecture de /content/drive/MyDrive/files/data/raw/zea_mays/ncbi_refseq_zea_b73_v5/Zm-B73-REFERENCE-NAM-5.0_GCF_902167145.1/GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_feature_table.txt.gz ...
Lecture de /content/drive/MyDrive/files/data/raw/zea_mays/uniprot_idmap_zea_mays/rolling_2026-10-11/uniprot_idm

In [19]:
#Cellule 5 — aperçu des résultats :
import pandas as pd, glob
for path in sorted(glob.glob(f"{DATA_DIR}/id_map_*.csv")):
    df = pd.read_csv(path)
    print(path, "-", len(df), "lignes")
    display(df["mapping_status"].value_counts())

/content/drive/MyDrive/files/data/id_map_nicotiana_tabacum_ASM71507v2_K326.csv - 79372 lignes


,count
mapping_status,
unmapped,79056
exact,316


/content/drive/MyDrive/files/data/id_map_nicotiana_tabacum_Ntab-TN90.csv - 74228 lignes


,count
mapping_status,
unmapped,73910
exact,318


/content/drive/MyDrive/files/data/id_map_oryza_sativa_IRGSP-1.0.csv - 35223 lignes


,count
mapping_status,
unmapped,34674
exact,547
ambiguous,2


/content/drive/MyDrive/files/data/id_map_zea_mays_Zm-B73-REFERENCE-NAM-5.0.csv - 49888 lignes


,count
mapping_status,
unmapped,49411
exact,475
ambiguous,2


In [20]:
#Cellule 6 — README Phase 2 :
!python write_phase_readme.py sources.yaml "{DATA_DIR}/manifest.csv" --phase 2 --out "{DATA_DIR}/README_phase2.md"

-> /content/drive/MyDrive/files/data/README_phase2.md (3 fichiers, 0.3 Mo, phase 2)
